# 07 — So sánh và thảo luận kết quả MNIST

Tổng hợp **Gaussian NB tự viết, Softmax tự viết, MLP tự viết và Stacking AdaBoost** từ kết quả các notebook trước. Không huấn luyện lại, không chọn cấu hình theo test. Kết quả sklearn chỉ dùng đối chiếu; các base model dùng riêng trong stacking không thay thế bốn phương pháp chính.

**Trước khi chạy:** chạy xong 02, 03, 04 và 06 (06 cần kết quả 05). Các thư mục `results/gaussian`, `results/softmax`, `results/mlp`, `results/stacking_adaboost` phải có `metrics.csv`, `config.json` và confusion matrix đã xuất. Chạy từ thư mục gốc hoặc `notebooks/`.

**Đầu ra:** bảng CSV và hình PNG 300 dpi tại `results/comparison/` để đưa vào report LaTeX và slide. Các nhận xét có số liệu tự cập nhật khi chạy; thảo luận chi tiết dành cho report.


## 1. Đọc kết quả của bốn phương pháp


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUTPUT = ROOT / 'results' / 'comparison'
OUTPUT.mkdir(parents=True, exist_ok=True)
METHODS = ['Gaussian NB', 'Softmax', 'MLP', 'Stacking AdaBoost']
SPLITS = ['Train', 'Validation', 'Test']
METRICS = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro']
sources = {
    'Gaussian NB': ('gaussian', 'GaussianNB tự viết', 'confusion_'),
    'Softmax': ('softmax', 'Softmax tự viết', 'confusion_'),
    'MLP': ('mlp', 'MLP tự viết', 'manual_confusion_'),
    'Stacking AdaBoost': ('stacking_adaboost', 'Stacking AdaBoost', 'Stacking_AdaBoost_confusion_'),
}
raw, configs, matrices, parts = {}, {}, {}, []
for method, (folder, model_name, cm_prefix) in sources.items():
    directory = ROOT / 'results' / folder
    raw[method] = pd.read_csv(directory / 'metrics.csv')
    configs[method] = json.loads((directory / 'config.json').read_text(encoding='utf-8'))
    table = raw[method].loc[raw[method].model == model_name].copy()
    table['method'] = method
    parts.append(table)
    for split in SPLITS:
        matrices[method, split] = pd.read_csv(directory / f'{cm_prefix}{split.lower()}.csv', index_col=0).to_numpy()
results = pd.concat(parts, ignore_index=True)
results = results[['method', 'split', *METRICS, 'fit_seconds', 'predict_seconds']]
print('Đã đọc kết quả:', ', '.join(METHODS))


## 2. Kiểm tra tính nhất quán và giao thức

Chỉ kiểm tra các điều kiện trực tiếp ảnh hưởng kết luận: đủ 4 phương pháp × 3 split, cùng số mẫu/seed, ma trận có kích thước đúng và các độ đo khớp confusion matrix. Các kiểm tra không chứng minh cùng chỉ số chia tập hoặc cùng phần cứng.

Theo các notebook đã chuẩn hóa, 02–04 chia cùng `test_size=10000, stratify=y, random_state=42` trên cùng thứ tự dữ liệu; 06 dùng chỉ số từ 05. Nếu thay cách chia hoặc dữ liệu đầu vào, cần chạy lại các kết quả liên quan trước khi so sánh.


In [ ]:
assert len(results) == 12 and not results.duplicated(['method', 'split']).any()
assert set(results.method) == set(METHODS) and set(results.split) == set(SPLITS)
protocol_rows = []
for method in METHODS:
    counts = {}
    for split in SPLITS:
        cm = matrices[method, split]
        assert cm.shape == (10, 10) and (cm >= 0).all()
        tp = np.diag(cm)
        precision = np.divide(tp, cm.sum(axis=0), out=np.zeros(10, dtype=float), where=cm.sum(axis=0) != 0)
        recall = np.divide(tp, cm.sum(axis=1), out=np.zeros(10, dtype=float), where=cm.sum(axis=1) != 0)
        f1 = np.divide(2*precision*recall, precision+recall, out=np.zeros(10, dtype=float), where=precision+recall != 0)
        measured = results.loc[(results.method == method) & (results.split == split), METRICS].iloc[0].to_numpy(dtype=float)
        expected = [tp.sum()/cm.sum(), precision.mean(), recall.mean(), f1.mean()]
        assert np.allclose(measured, expected, atol=1e-8, rtol=1e-8), f'CSV và confusion matrix không khớp: {method}, {split}'
        counts[split] = int(cm.sum())
    protocol_rows.append(dict(method=method, seed=configs[method]['seed'], train=counts['Train'],
        validation=counts['Validation'], test=counts['Test'], dtype=configs[method].get('dtype', 'Không ghi trong config')))
protocol = pd.DataFrame(protocol_rows).set_index('method').loc[METHODS]
assert (protocol[['train', 'validation', 'test']].to_numpy() == [50000, 10000, 10000]).all()
assert protocol.seed.nunique() == 1 and protocol.seed.iloc[0] == 42
display(protocol)


**Phạm vi so sánh:** cùng số mẫu, cách scale pixel và độ đo; kiến trúc, dtype, cách chọn siêu tham số và ngân sách tìm kiếm không hoàn toàn giống nhau. Vì vậy đây là so sánh các pipeline đã xây dựng, không phải benchmark kiểm soát mọi yếu tố. Giữ nguyên quyết định chọn cấu hình từ các notebook trước.


## 3. Chất lượng phân lớp trên test

Bảng chứa đủ accuracy, precision macro, recall macro và F1 macro. Số ảnh đúng/sai lấy trực tiếp từ confusion matrix. Xếp theo accuracy chỉ để mô tả kết quả, không dùng test để chọn lại mô hình.


In [ ]:
test = results[results.split == 'Test'].set_index('method').loc[METHODS].copy()
test['correct'] = [int(np.trace(matrices[m, 'Test'])) for m in METHODS]
test['errors'] = protocol['test'] - test['correct']
ranking = test[METRICS + ['correct', 'errors']].sort_values(['accuracy', 'f1_macro'], ascending=False)
display(ranking.round(5))
best_acc = ranking.index[0]
best_f1 = test.f1_macro.idxmax()
display(Markdown(f'**Kết quả:** {best_acc} có test accuracy cao nhất trong bốn pipeline (**{test.loc[best_acc, "accuracy"]:.2%}**); '
                 f'{best_f1} có F1 macro cao nhất (**{test.loc[best_f1, "f1_macro"]:.4f}**). '
                 'Đây là kết quả trên một split/seed; chênh lệch nhỏ chưa chứng minh khác biệt có ý nghĩa thống kê.'))


## 4. Chênh lệch train–validation–test

Bảng và biểu đồ dùng cùng mô hình cuối ở cả ba tập. Train của stacking là in-sample sau refit, không phải ước lượng độc lập bằng OOF. Validation của từng phương pháp đã dùng chọn cấu hình nên có thể lạc quan. Không kết luận overfitting chỉ từ một khoảng cách điểm số.


In [ ]:
accuracy = results.pivot(index='method', columns='split', values='accuracy').loc[METHODS, SPLITS]
f1_table = results.pivot(index='method', columns='split', values='f1_macro').loc[METHODS, SPLITS]
gaps = accuracy.copy()
gaps['train_minus_validation_pp'] = (gaps.Train - gaps.Validation) * 100
gaps['validation_minus_test_pp'] = (gaps.Validation - gaps.Test) * 100
display(gaps.round(4))
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, table, title in zip(axes, [accuracy, f1_table], ['Accuracy', 'F1 macro']):
    table.plot.bar(ax=ax, rot=15, ylim=(0, 1))
    ax.set(xlabel='', ylabel='Điểm số', title=f'{title} trên ba tập')
fig.tight_layout(); fig.savefig(OUTPUT / 'scores_by_split.png', dpi=300); plt.show()


**Cần thảo luận trong report:** mô hình nào có điểm train cao nhưng validation thấp hơn đáng kể? Mô hình nào thấp trên cả hai tập? Liên hệ với năng lực biểu diễn và ngân sách huấn luyện, nhưng trình bày nguyên nhân như giả thuyết nếu chưa có thí nghiệm kiểm chứng. “Điểm phần trăm” là hiệu hai tỷ lệ nhân 100, không phải phần trăm cải thiện tương đối.


## 5. Chi phí huấn luyện và dự đoán

- `fit_seconds`: một lần fit cấu hình cuối. Với stacking, gồm tạo OOF (fit và dự đoán trong fold), refit các base model và fit meta-model cuối.
- Thời gian khảo sát được ghi riêng. Với Gaussian NB, Softmax và MLP, khảo sát đã bao gồm lần fit mô hình cuối được tái sử dụng; **không cộng máy móc hai cột**.
- Thời gian tìm kiếm của stacking bên dưới chỉ là meta-search, chưa gồm notebook 05. Stacking phải chịu thêm chi phí lựa chọn/tuning base models nếu tính toàn bộ dự án.
- Dự đoán stacking/voting gồm tạo xác suất base models và bước gộp, không chỉ riêng `meta.predict`.

Các phép đo chỉ thực hiện một lần, có thể từ các lượt chạy hoặc môi trường khác nhau; không kết luận tốc độ tuyệt đối cho mọi máy. Nếu bật cache ở 06, thời gian fit là chi phí tính đã ghi trước đó; wall time lần chạy lại nằm trong config riêng.


In [ ]:
search_times = {
    'Gaussian NB': configs['Gaussian NB']['search_fit_seconds'],
    'Softmax': configs['Softmax']['search_seconds'],
    'MLP': configs['MLP']['search_fit_seconds'],
    'Stacking AdaBoost': configs['Stacking AdaBoost']['meta_search_fit_seconds'],
}
timing = test[['fit_seconds', 'predict_seconds']].copy()
timing['search_fit_seconds'] = pd.Series(search_times)
timing['test_samples'] = protocol['test']
timing['test_samples_per_second'] = timing.test_samples / timing.predict_seconds
timing['search_scope'] = ['Smoothing; gồm cấu hình cuối', 'LR và L2; gồm cấu hình cuối',
                          'LR và kiến trúc; gồm cấu hình cuối', 'Chỉ meta-search; chưa tính notebook 05']
assert (timing[['fit_seconds', 'predict_seconds']] > 0).all().all()
display(timing.round(4))
print('Cache ở notebook 06:', configs['Stacking AdaBoost']['cache_enabled'])
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, column, title in zip(axes, ['fit_seconds', 'predict_seconds'], ['Fit cấu hình cuối', 'Dự đoán 10.000 ảnh test']):
    timing[column].plot.bar(ax=ax, rot=15, logy=True, color='steelblue')
    ax.set(xlabel='', ylabel='Giây — thang log', title=title)
fig.tight_layout(); fig.savefig(OUTPUT / 'timing.png', dpi=300); plt.show()


## 6. Đối chiếu cài đặt tự viết với sklearn

Gaussian NB đối chiếu cùng epsilon; Softmax đối chiếu mức phạt L2 tương ứng nhưng khác optimizer; MLP cùng kiến trúc/SGD/L2=0 nhưng khác khởi tạo và shuffle. Gần nhau về accuracy không thay thế kiểm tra công thức/gradient. Stacking dùng thư viện theo phương án ensemble của nhóm nên không có cặp “tự viết–sklearn” riêng.


In [ ]:
reference_names = {'Gaussian NB': 'GaussianNB sklearn', 'Softmax': 'Softmax sklearn', 'MLP': 'MLP sklearn'}
reference_rows = []
for method, reference_name in reference_names.items():
    ref = raw[method].loc[(raw[method].model == reference_name) & (raw[method].split == 'Test')].iloc[0]
    reference_rows.append(dict(method=method, manual_accuracy=test.loc[method, 'accuracy'], sklearn_accuracy=ref.accuracy,
        accuracy_difference_pp=(test.loc[method, 'accuracy']-ref.accuracy)*100,
        manual_f1=test.loc[method, 'f1_macro'], sklearn_f1=ref.f1_macro))
reference_check = pd.DataFrame(reference_rows).set_index('method')
display(reference_check.round(5))


## 7. Lớp khó và các cặp nhầm chính

Không lặp bốn confusion matrix lớn đã có ở từng notebook. Dùng recall theo chữ số để so sánh lỗi bỏ sót giữa các phương pháp; bảng cặp nhầm chỉ lấy 3 cặp có số lượng lớn nhất mỗi mô hình. Bảng đầy đủ precision/recall/F1 theo lớp được xuất CSV.


In [ ]:
class_rows, confusion_rows = [], []
for method in METHODS:
    cm = matrices[method, 'Test']
    tp = np.diag(cm)
    precision = np.divide(tp, cm.sum(axis=0), out=np.zeros(10, dtype=float), where=cm.sum(axis=0) != 0)
    recall = np.divide(tp, cm.sum(axis=1), out=np.zeros(10, dtype=float), where=cm.sum(axis=1) != 0)
    f1 = np.divide(2*precision*recall, precision+recall, out=np.zeros(10, dtype=float), where=precision+recall != 0)
    for digit in range(10):
        class_rows.append(dict(method=method, digit=digit, precision=precision[digit], recall=recall[digit], f1=f1[digit], support=int(cm[digit].sum())))
    pairs = [(i, j, int(cm[i, j])) for i in range(10) for j in range(10) if i != j and cm[i, j] > 0]
    for i, j, count in sorted(pairs, key=lambda row: (-row[2], row[0], row[1]))[:3]:
        confusion_rows.append(dict(method=method, true_digit=i, predicted_digit=j, count=count, fraction_of_true_class=count/cm[i].sum()))
per_class = pd.DataFrame(class_rows)
recall_table = per_class.pivot(index='method', columns='digit', values='recall').loc[METHODS]
top_confusions = pd.DataFrame(confusion_rows)
fig, ax = plt.subplots(figsize=(11, 4))
im = ax.imshow(recall_table.to_numpy(), vmin=0, vmax=1, cmap='Blues', aspect='auto')
ax.set(xticks=np.arange(10), xticklabels=np.arange(10), yticks=np.arange(4), yticklabels=METHODS,
       xlabel='Nhãn thật', title='Recall trên test theo chữ số')
for i in range(4):
    for j in range(10):
        value = recall_table.iloc[i, j]
        ax.text(j, i, f'{value:.1%}', ha='center', va='center', color='white' if value > .6 else 'black', fontsize=9)
fig.colorbar(im, ax=ax, label='Recall'); fig.tight_layout()
fig.savefig(OUTPUT / 'recall_by_digit.png', dpi=300); plt.show()
display(top_confusions.round(4))


**Diễn giải:** phân biệt “lớp recall thấp” với “cặp nhầm có số lượng lớn”. Các lớp có số lượng ảnh khác nhau, nên đọc cùng tỷ lệ trên số ảnh lớp thật. Không suy ra nguyên nhân chỉ từ bảng nhầm; dùng ảnh sai ở notebook mô hình để minh họa trong report.


## 8. Ensemble có thực sự cải thiện?

So sánh đúng ba base model đã chọn, soft voting và stacking trên cùng test. Chênh lệch được báo cáo dù dương hay âm. Không đổi bộ ba hoặc meta-parameters để tăng điểm sau khi xem test.


In [ ]:
stacking_cfg = configs['Stacking AdaBoost']
ensemble_names = stacking_cfg['model_order'] + ['Soft voting', 'Stacking AdaBoost']
ensemble_test = raw['Stacking AdaBoost'].query("split == 'Test'").set_index('model').loc[ensemble_names, METRICS + ['fit_seconds', 'predict_seconds']]
display(ensemble_test.round(5))
best_base = ensemble_test.loc[stacking_cfg['model_order'], 'accuracy'].max()
stack_acc = ensemble_test.loc['Stacking AdaBoost', 'accuracy']
vote_acc = ensemble_test.loc['Soft voting', 'accuracy']
ensemble_gains = pd.DataFrame([
    {'comparison': 'Stacking − base tốt nhất', 'accuracy_difference_pp': (stack_acc-best_base)*100},
    {'comparison': 'Stacking − soft voting', 'accuracy_difference_pp': (stack_acc-vote_acc)*100},
])
display(ensemble_gains.round(4))
message = 'Stacking chưa vượt cả hai baseline về accuracy.' if stack_acc <= max(best_base, vote_acc) else 'Stacking vượt cả hai baseline về accuracy trong lần chạy này.'
display(Markdown('**Nhận xét:** ' + message + ' Không suy ra khác biệt có ý nghĩa thống kê chỉ từ một lần đánh giá.'))


**Nếu stacking thấp hơn:** thảo luận như các khả năng cần kiểm chứng: bộ ba được chọn theo voting nên chưa chắc phù hợp meta-learner; cây meta nông có thể chưa khai thác tốt xác suất; phân phối đặc trưng OOF và base refit có thể khác nhau; xác suất giữa các họ mô hình chưa được hiệu chỉnh chung. Không viết các khả năng này như nguyên nhân đã được chứng minh.

**Nếu stacking cao hơn:** nêu mức tăng cùng chi phí huấn luyện/dự đoán tăng thêm. Chưa có kiểm định cặp hoặc nhiều seed, nên tránh khẳng định ưu thế ổn định.


## 9. Ưu/nhược điểm và khả năng giải thích

Độ chính xác và thời gian lấy từ các bảng định lượng phía trên. Bảng dưới bổ sung đặc điểm phương pháp, không dùng nhãn “tốt nhất” chung cho mọi tiêu chí.


In [ ]:
discussion = pd.DataFrame([
    ['Gaussian NB', 'Thống kê mean/variance/prior theo lớp; không tối ưu lặp', 'Tham số có ý nghĩa xác suất, nhưng 784 pixel khó diễn giải riêng lẻ', 'Giả định Gaussian và độc lập có điều kiện hạn chế mô tả ảnh'],
    ['Softmax', 'Mô hình tuyến tính; có thể trực quan trọng số theo chữ số', 'Đóng góp tuyến tính vào logit; trọng số không phải quan hệ nhân quả', 'Khó biểu diễn quan hệ phi tuyến và cấu trúc không gian'],
    ['MLP', 'Học biểu diễn phi tuyến qua các lớp ẩn', 'Khó giải thích trực tiếp từng quyết định hơn mô hình tuyến tính', 'Nhiều tham số; phụ thuộc kiến trúc, learning rate và quá trình tối ưu'],
    ['Stacking AdaBoost', 'Kết hợp xác suất của nhiều họ mô hình', 'Có thể xem vai trò các đặc trưng xác suất, nhưng phụ thuộc toàn pipeline', 'OOF/refit tốn chi phí; không bảo đảm hơn base model hoặc voting'],
], columns=['method', 'strength', 'interpretability', 'limitation'])
display(discussion)


## 10. Những việc cần chốt trong report và slide

- [ ] Nêu bốn phương pháp, công thức/pseudocode và phần tự cài đặt; ghi rõ phần ensemble dùng thư viện theo hướng dẫn của giảng viên.
- [ ] Mô tả preprocessing, split, seed, cấu hình cuối và tiêu chí chọn tham số; ghi nhận ngân sách khảo sát không giống nhau.
- [ ] Trình bày đủ bốn độ đo và confusion matrix trên ba tập; chọn hình chính, đưa bảng lớn vào phụ lục.
- [ ] Dùng biểu đồ siêu tham số từ 02–04/06; giải thích bằng kết quả thực tế, không chỉ chép hình.
- [ ] So sánh độ chính xác, thời gian fit/predict và khả năng giải thích; ghi rõ cache, phần cứng và cảnh báo hội tụ khi có.
- [ ] Giải thích kết quả ensemble dù không vượt baseline; phân biệt OOF training, train in-sample và test độc lập.
- [ ] Nêu giới hạn: một split/seed, chưa kiểm định chênh lệch nhỏ, không có nested CV cho toàn quy trình chọn/tuning.
- [ ] Hoàn thiện tài liệu tham khảo, phân công/tỷ lệ đóng góp và khai báo sử dụng AI theo yêu cầu đề.
- [ ] Hoàn thiện README, report PDF và slide; thử chạy lại từ đầu trong môi trường cài theo requirements.

Notebook 07 cung cấp bảng tổng hợp; nó không tự chứng minh công thức đúng hoặc thay thế phần lập luận của report. Không cần thêm PCA/CNN/demo để đáp ứng phần so sánh bắt buộc.


## 11. Xuất bảng và hình cho báo cáo

Các độ đo lưu dạng tỷ lệ [0, 1], cột có hậu tố `_pp` là điểm phần trăm, thời gian là giây. Chạy lại ghi đè các file cùng tên.


In [ ]:
tables = {'metrics_all_splits': results, 'protocol': protocol, 'test_ranking': ranking,
          'generalization_gaps': gaps, 'timing': timing, 'sklearn_comparison': reference_check,
          'per_class_test': per_class, 'top_confusions': top_confusions, 'ensemble_test': ensemble_test,
          'ensemble_gains': ensemble_gains, 'discussion': discussion}
indexed = {'protocol', 'test_ranking', 'generalization_gaps', 'timing', 'sklearn_comparison', 'ensemble_test'}
for name, table in tables.items():
    table.to_csv(OUTPUT / f'{name}.csv', index=name in indexed)
manifest = {'methods': METHODS, 'splits': SPLITS, 'source_folders': {m: sources[m][0] for m in METHODS},
            'purpose': 'tong hop ket qua da chot, khong tuning tren test',
            'aggregation_versions': {'numpy': np.__version__, 'pandas': pd.__version__, 'matplotlib': plt.matplotlib.__version__}}
(OUTPUT / 'comparison_config.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print('Đã lưu bảng và hình tại:', OUTPUT)
